In [ ]:
# ============================================================
# Regime Filter Analysis — Market Regime Classification
# ============================================================
# Uncomment and run these if packages are not installed:

# !pip install yfinance
# !pip install numpy
# !pip install pandas
# !pip install matplotlib

In [ ]:
import sys, os
# Ensure lib/ is importable (for Google Sheets logging)
for _p in ['/content/trading-strategies', os.path.join(os.getcwd(), '..')]:
    if os.path.isdir(os.path.join(_p, 'lib')) and _p not in sys.path:
        sys.path.insert(0, _p)

# ============================================================
# Cell 1 — Imports + Inline Regime Filter Functions
# ============================================================
# Self-contained for Google Colab — no external lib imports needed.

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
from matplotlib.colors import ListedColormap
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------------
# Regime Filter Functions (inline copy of lib/regime_filter.py)
# ------------------------------------------------------------------

def realized_volatility(close, window=21):
    """
    Annualized realized volatility over a trailing window.
    Uses log returns so it compounds correctly.
    """
    log_rets = np.log(close / close.shift(1))
    return log_rets.rolling(window=window).std() * np.sqrt(252)


def classify_regimes(close, trend_period=200, vol_window=21, vol_lookback=126):
    """
    Classify each day into one of 4 regimes using ONLY backward-looking data.

    CRITICAL: All indicators are shifted by 1 bar so the regime label on
    day T is computed from data through day T-1. This means you can safely
    use regime[T] to filter signals on day T with ZERO lookahead bias.

    Two dimensions:
      1. TREND:      BULL (prev close > prev SMA) / BEAR
      2. VOLATILITY:  HIGH (prev rvol > rolling median of rvol) / LOW

    Combined into 4 regimes:
      - BULL_CALM:  Trending up, low vol
      - BULL_WILD:  Trending up, high vol
      - BEAR_CALM:  Trending down, low vol
      - BEAR_WILD:  Crisis mode, high vol

    Parameters
    ----------
    close : pd.Series
        Daily close prices with DatetimeIndex.
    trend_period : int
        SMA period for trend classification (default 200 ~ 1 year).
    vol_window : int
        Rolling window for realized volatility (default 21 ~ 1 month).
    vol_lookback : int
        Rolling window for vol median (default 126 ~ 6 months).
        Vol is HIGH if current realized vol > rolling median.

    Returns
    -------
    pd.DataFrame
        Columns: close, sma, trend, realized_vol, vol_median,
                 vol_regime, regime, regime_code
    """
    close = close.astype(float).squeeze()
    close.name = 'close'

    # Trend: price vs SMA — shifted 1 bar (yesterday's close vs yesterday's SMA)
    sma = close.rolling(window=trend_period).mean()
    prev_close = close.shift(1)
    prev_sma = sma.shift(1)
    trend = pd.Series(
        np.where(prev_close > prev_sma, 'BULL', 'BEAR'),
        index=close.index, name='trend'
    )

    # Volatility: realized vol vs its own rolling median — all shifted 1 bar
    rvol = realized_volatility(close, window=vol_window).shift(1)
    vol_median = rvol.rolling(window=vol_lookback).median()
    vol_regime = pd.Series(
        np.where(rvol > vol_median, 'HIGH', 'LOW'),
        index=close.index, name='vol_regime'
    )

    # Combined regime label
    regime = pd.Series(
        np.where(
            (trend == 'BULL') & (vol_regime == 'LOW'), 'BULL_CALM',
            np.where(
                (trend == 'BULL') & (vol_regime == 'HIGH'), 'BULL_WILD',
                np.where(
                    (trend == 'BEAR') & (vol_regime == 'LOW'), 'BEAR_CALM',
                    'BEAR_WILD'
                )
            )
        ),
        index=close.index, name='regime'
    )

    regime_code = regime.map({
        'BULL_CALM': 0, 'BULL_WILD': 1,
        'BEAR_CALM': 2, 'BEAR_WILD': 3
    }).astype(float)

    return pd.DataFrame({
        'close': close,
        'sma': sma,
        'trend': trend,
        'realized_vol': rvol,
        'vol_median': vol_median,
        'vol_regime': vol_regime,
        'regime': regime,
        'regime_code': regime_code
    })


print('Regime filter functions loaded.')
print('  - realized_volatility(close, window=21)')
print('  - classify_regimes(close, trend_period=200, vol_window=21, vol_lookback=126)')

In [ ]:
# ============================================================
# Cell 2 — Configuration and Data Download
# ============================================================

TICKER      = 'SPY'
START_DATE  = '2015-01-01'

TREND_PERIOD  = 200    # SMA period for trend classification
VOL_WINDOW    = 21     # Realized vol window (~1 month)
VOL_LOOKBACK  = 126    # Rolling median lookback for vol (~6 months)

# --- Download ---
raw = yf.download(TICKER, start=START_DATE, auto_adjust=True)

# Handle MultiIndex columns (yfinance >= 0.2.18)
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

close = raw['Close']

print(f'\nTicker:     {TICKER}')
print(f'Start date: {START_DATE}')
print(f'Bars:       {len(close)}')
print(f'Date range: {close.index[0].date()} to {close.index[-1].date()}')
print(f'\nClose price (last 5):')
print(close.tail())

In [ ]:
# ============================================================
# Cell 3 — Classify Regimes and Print Distribution
# ============================================================

df = classify_regimes(
    close,
    trend_period=TREND_PERIOD,
    vol_window=VOL_WINDOW,
    vol_lookback=VOL_LOOKBACK
)

# Drop rows where regime is not yet defined (warmup period)
valid = df.dropna(subset=['regime', 'realized_vol', 'vol_median'])
# Also drop rows where regime might be set but vol_median is NaN
valid = valid[valid['vol_median'].notna()].copy()

print('=' * 65)
print('REGIME DISTRIBUTION')
print('=' * 65)

counts = valid['regime'].value_counts()
total = len(valid)

regime_order = ['BULL_CALM', 'BULL_WILD', 'BEAR_CALM', 'BEAR_WILD']
regime_emoji = {'BULL_CALM': '[+]', 'BULL_WILD': '[~]', 'BEAR_CALM': '[-]', 'BEAR_WILD': '[!]'}

print(f"\n  {'Regime':12s}  {'Count':>6s}  {'Pct':>6s}  Bar Chart")
print('  ' + '-' * 55)

for regime in regime_order:
    n = counts.get(regime, 0)
    pct = n / total * 100 if total > 0 else 0
    bar = '#' * int(pct / 2)
    tag = regime_emoji[regime]
    print(f'  {regime:12s}  {n:6d}  {pct:5.1f}%  {bar}  {tag}')

print(f'\n  Total classified: {total} days')
print(f'  Date range: {valid.index[0].date()} to {valid.index[-1].date()}')
print(f'  Warmup rows dropped: {len(df) - len(valid)}')

In [ ]:
# ============================================================
# Cell 4 — Per-Regime Buy & Hold Return Statistics (Annualized)
# ============================================================

daily_rets = valid['close'].pct_change()

print('=' * 72)
print('PER-REGIME RETURN STATISTICS (annualized, buy & hold in each regime)')
print('=' * 72)
print(f"  {'Regime':12s} {'Ann Ret':>10s} {'Ann Vol':>10s} {'Sharpe':>8s} {'Max DD':>8s} {'Days':>7s}")
print('  ' + '-' * 60)

for regime in ['BULL_CALM', 'BULL_WILD', 'BEAR_CALM', 'BEAR_WILD']:
    mask = valid['regime'] == regime
    subset = daily_rets[mask].dropna()
    if len(subset) < 5:
        print(f'  {regime:12s}  insufficient data')
        continue

    ann_ret = subset.mean() * 252
    ann_vol = subset.std() * np.sqrt(252)
    sharpe  = ann_ret / ann_vol if ann_vol > 0 else 0.0
    cum     = (1 + subset).cumprod()
    max_dd  = ((cum / cum.cummax()) - 1).min()

    print(f'  {regime:12s} {ann_ret:>9.1%} {ann_vol:>9.1%} {sharpe:>7.2f} {max_dd:>7.1%} {len(subset):>7d}')

print()
print('  INTERPRETATION')
print('  ' + '-' * 60)
print('  BULL_CALM  : Ideal for trend-following strategies (MACD, EMA,')
print('               Donchian). Go full position size.')
print('  BULL_WILD  : Trends still work but expect whipsaws. Reduce size.')
print('  BEAR_CALM  : Grinding down. Mean-reversion can work here.')
print('               Trend-following gets chopped up.')
print('  BEAR_WILD  : Crisis mode. STAY OUT. Capital preservation is the')
print('               only goal. Only crisis-alpha strategies survive.')

In [ ]:
# ============================================================
# Cell 5 — Price Chart Colored by Regime (3 Panels)
# ============================================================

REGIME_COLORS = {
    'BULL_CALM': '#00cc66',   # green
    'BULL_WILD': '#ff9900',   # orange
    'BEAR_CALM': '#3399ff',   # blue
    'BEAR_WILD': '#ff3333',   # red
}

BG_COLOR  = '#FFFFFF'
TXT_COLOR = '#5A6270'

fig, axes = plt.subplots(3, 1, figsize=(18, 12), sharex=True,
                          gridspec_kw={'height_ratios': [4, 2, 0.6]})
fig.patch.set_facecolor(BG_COLOR)

for ax in axes:
    ax.set_facecolor(BG_COLOR)
    ax.tick_params(colors=TXT_COLOR, labelsize=9)
    for spine in ax.spines.values():
        spine.set_color('#E2E5EB')

# --- Panel 1: Price + SMA with regime-colored background ---
ax1 = axes[0]
ax1.plot(valid.index, valid['close'], color='#1A1D23', linewidth=0.8, label='Close')
ax1.plot(valid.index, valid['sma'], color='gray', linewidth=0.7, linestyle='--', label=f'SMA({TREND_PERIOD})')

# Color background by regime
for regime, color in REGIME_COLORS.items():
    mask = valid['regime'] == regime
    if mask.any():
        ax1.fill_between(valid.index, valid['close'].min() * 0.95,
                         valid['close'].max() * 1.05,
                         where=mask, color=color, alpha=0.15, linewidth=0)

ax1.set_ylabel('Price ($)', color=TXT_COLOR, fontsize=11)
ax1.set_title(f'{TICKER} — Regime Filter Analysis', color='#1A1D23', fontsize=14, fontweight='bold', pad=12)

# Legend with colored patches
legend_patches = [
    mpatches.Patch(color=REGIME_COLORS['BULL_CALM'], alpha=0.5, label='BULL_CALM'),
    mpatches.Patch(color=REGIME_COLORS['BULL_WILD'], alpha=0.5, label='BULL_WILD'),
    mpatches.Patch(color=REGIME_COLORS['BEAR_CALM'], alpha=0.5, label='BEAR_CALM'),
    mpatches.Patch(color=REGIME_COLORS['BEAR_WILD'], alpha=0.5, label='BEAR_WILD'),
]
ax1.legend(handles=legend_patches, loc='upper left', fontsize=9,
           facecolor=BG_COLOR, edgecolor='#E2E5EB', labelcolor=TXT_COLOR)

# --- Panel 2: Realized Volatility vs Median ---
ax2 = axes[1]
ax2.plot(valid.index, valid['realized_vol'], color='#ff4444', linewidth=0.8, label='Realized Vol')
ax2.plot(valid.index, valid['vol_median'], color='#4488ff', linewidth=0.8,
         linestyle='--', label=f'Vol Median ({VOL_LOOKBACK}d)')
ax2.fill_between(valid.index, valid['realized_vol'], valid['vol_median'],
                 where=valid['realized_vol'] > valid['vol_median'],
                 color='#ff4444', alpha=0.2, label='Vol > Median (HIGH)')
ax2.fill_between(valid.index, valid['realized_vol'], valid['vol_median'],
                 where=valid['realized_vol'] <= valid['vol_median'],
                 color='#4488ff', alpha=0.2, label='Vol <= Median (LOW)')
ax2.set_ylabel('Annualized Vol', color=TXT_COLOR, fontsize=11)
ax2.legend(loc='upper right', fontsize=8, facecolor=BG_COLOR,
           edgecolor='#E2E5EB', labelcolor=TXT_COLOR)

# --- Panel 3: Regime Timeline Strip ---
ax3 = axes[2]
regime_num = valid['regime'].map({
    'BULL_CALM': 0, 'BULL_WILD': 1, 'BEAR_CALM': 2, 'BEAR_WILD': 3
}).values

cmap = ListedColormap([REGIME_COLORS['BULL_CALM'], REGIME_COLORS['BULL_WILD'],
                        REGIME_COLORS['BEAR_CALM'], REGIME_COLORS['BEAR_WILD']])

ax3.imshow(regime_num.reshape(1, -1), aspect='auto', cmap=cmap,
           extent=[mdates.date2num(valid.index[0]), mdates.date2num(valid.index[-1]), 0, 1],
           vmin=0, vmax=3)
ax3.set_yticks([])
ax3.set_ylabel('Regime', color=TXT_COLOR, fontsize=10)
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax3.xaxis.set_major_locator(mdates.YearLocator())

plt.tight_layout()
plt.show()
print('Chart: 3-panel regime overlay.')

In [ ]:
# ============================================================
# Cell 6 — Monthly Regime Calendar Heatmap
# ============================================================

# Determine the dominant regime for each month
valid_monthly = valid[['regime']].copy()
valid_monthly['year']  = valid_monthly.index.year
valid_monthly['month'] = valid_monthly.index.month

# Dominant regime = most frequent regime in each month
monthly_regime = (
    valid_monthly
    .groupby(['year', 'month'])['regime']
    .agg(lambda x: x.value_counts().idxmax())
    .reset_index()
)

years = sorted(monthly_regime['year'].unique())
months = list(range(1, 13))
month_labels = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

regime_to_num = {'BULL_CALM': 0, 'BULL_WILD': 1, 'BEAR_CALM': 2, 'BEAR_WILD': 3}
regime_to_letter = {'BULL_CALM': 'C', 'BULL_WILD': 'W', 'BEAR_CALM': 'C', 'BEAR_WILD': 'W'}

# Build grid
grid = np.full((len(years), 12), np.nan)
letters = [['' for _ in range(12)] for _ in range(len(years))]

for _, row in monthly_regime.iterrows():
    yi = years.index(row['year'])
    mi = row['month'] - 1
    grid[yi, mi] = regime_to_num[row['regime']]
    letters[yi][mi] = regime_to_letter[row['regime']]

# Plot
BG_COLOR  = '#FFFFFF'
TXT_COLOR = '#5A6270'

fig, ax = plt.subplots(figsize=(14, max(4, len(years) * 0.55)))
fig.patch.set_facecolor(BG_COLOR)
ax.set_facecolor(BG_COLOR)

cmap = ListedColormap([REGIME_COLORS['BULL_CALM'], REGIME_COLORS['BULL_WILD'],
                        REGIME_COLORS['BEAR_CALM'], REGIME_COLORS['BEAR_WILD']])

im = ax.imshow(grid, cmap=cmap, vmin=0, vmax=3, aspect='auto')

# Add letters in each cell
for yi in range(len(years)):
    for mi in range(12):
        if letters[yi][mi]:
            txt_c = 'white' if not np.isnan(grid[yi, mi]) else '#8C95A3'
            ax.text(mi, yi, letters[yi][mi], ha='center', va='center',
                    fontsize=10, fontweight='bold', color=txt_c)

ax.set_xticks(range(12))
ax.set_xticklabels(month_labels, color=TXT_COLOR, fontsize=10)
ax.set_yticks(range(len(years)))
ax.set_yticklabels(years, color=TXT_COLOR, fontsize=10)
ax.tick_params(length=0)

for spine in ax.spines.values():
    spine.set_visible(False)

ax.set_title(f'{TICKER} — Monthly Regime Calendar  (C = Calm, W = Wild)',
             color='#1A1D23', fontsize=13, fontweight='bold', pad=14)

# Legend
legend_patches = [
    mpatches.Patch(color=REGIME_COLORS['BULL_CALM'], label='BULL_CALM (C)'),
    mpatches.Patch(color=REGIME_COLORS['BULL_WILD'], label='BULL_WILD (W)'),
    mpatches.Patch(color=REGIME_COLORS['BEAR_CALM'], label='BEAR_CALM (C)'),
    mpatches.Patch(color=REGIME_COLORS['BEAR_WILD'], label='BEAR_WILD (W)'),
]
ax.legend(handles=legend_patches, loc='upper center',
          bbox_to_anchor=(0.5, -0.08), ncol=4, fontsize=9,
          facecolor=BG_COLOR, edgecolor='#E2E5EB', labelcolor=TXT_COLOR)

plt.tight_layout()
plt.show()
print('Heatmap: monthly dominant regime calendar.')

In [ ]:
# ============================================================
# Cell 7 — Integration Guide + Current Regime
# ============================================================

# --- Current regime ---
latest = df.dropna(subset=['regime']).iloc[-1]
latest_date = df.dropna(subset=['regime']).index[-1]

print('=' * 70)
print(f'CURRENT REGIME for {TICKER}')
print('=' * 70)
print(f'  Date:          {latest_date.date()}')
print(f'  Close:         ${latest["close"]:.2f}')
print(f'  SMA({TREND_PERIOD}):     ${latest["sma"]:.2f}')
print(f'  Trend:         {latest["trend"]}')
print(f'  Realized Vol:  {latest["realized_vol"]:.1%}')
print(f'  Vol Median:    {latest["vol_median"]:.1%}')
print(f'  Vol Regime:    {latest["vol_regime"]}')
print(f'  >>> REGIME:    {latest["regime"]} <<<')
print()

print('=' * 70)
print('HOW TO USE IN STRATEGY NOTEBOOKS')
print('=' * 70)
print()
print('  Step 1: Classify regimes on your price series')
print('  -----------------------------------------------')
print('    regimes = classify_regimes(close, trend_period=200,')
print('                              vol_window=21, vol_lookback=126)')
print()
print('  Step 2: Filter your entry signals by regime')
print('  -----------------------------------------------')
print()
print('    # For TREND-FOLLOWING strategies (MACD, EMA, Donchian):')
print('    # Only trade when in BULL_CALM or BULL_WILD')
print('    bull_mask = regimes["regime"].isin(["BULL_CALM", "BULL_WILD"])')
print('    entries = entries & bull_mask')
print()
print('    # For MEAN-REVERSION strategies (RSI, Bollinger):')
print('    # Only trade when in CALM regimes (low volatility)')
print('    calm_mask = regimes["regime"].isin(["BULL_CALM", "BEAR_CALM"])')
print('    entries = entries & calm_mask')
print()
print('    # AVOID BEAR_WILD at all costs — crisis mode.')
print('    # no_crisis = regimes["regime"] != "BEAR_WILD"')
print('    # entries = entries & no_crisis')
print()
print('  Expected impact of regime filtering:')
print('  -----------------------------------------------')
print('    - Slightly lower absolute return (you skip some days)')
print('    - MUCH better Sharpe ratio (remove worst risk-adjusted days)')
print('    - Significantly smaller max drawdown')
print('    - Fewer trades, but higher quality trades')
print()
print('=' * 70)
print('WHY THIS WORKS')
print('=' * 70)
print()
print('  1. VOLATILITY CLUSTERING (Cont, 2001):')
print('     High-vol days cluster together. If yesterday was high-vol,')
print('     today is likely high-vol too. The regime filter exploits')
print('     this well-documented stylized fact of asset returns.')
print()
print('  2. NO LOOKAHEAD BIAS:')
print('     All indicators are shifted by 1 bar. The regime on day T')
print('     uses only data through day T-1. This means the backtest')
print('     results are realistic and achievable in live trading.')
print()
print('  3. REGIME PERSISTENCE:')
print('     Markets tend to stay in regimes for weeks to months.')
print('     This is not a high-frequency signal — it is a slow filter')
print('     that keeps you on the right side of the market.')
print()
print('  4. STRATEGY-REGIME ALIGNMENT:')
print('     Trend-following profits from persistent moves (BULL).')
print('     Mean-reversion profits from range-bound markets (CALM).')
print('     Matching strategy type to regime type is the key insight.')